## enhance the rag built (chatpdf) with gemini and langchain


In [10]:
import os
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts.chat import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)

In [11]:
load_dotenv()  # Loads from .env

api_key = os.getenv("GOOGLE_API_KEY")
print("API Key loaded:", bool(api_key))  # Just to confirm, prints True if loaded


llm = init_chat_model("gemini-2.0-flash", model_provider="google_genai")


embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

API Key loaded: True


In [12]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Enable loading of pickle file safely
vector_store = FAISS.load_local(
    "D:/arc-2025/arc/ai/Uddav-Rajbhandari/Rag_finetunngLLM and prompt/source codes/saved_chatpdf/vectorstore", 
    embed_model, 
    allow_dangerous_deserialization=True
)


In [13]:
# Retriever from the vectorstore - controls number of docs retrieved
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

# Conversation memory to maintain chat history for context
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# Custom prompt to ensure answers are grounded only in retrieved context
system_prompt = """
You are a helpful AI assistant that answers user questions using only the provided context.
If the context does not contain the answer, respond with: "I do not have context for that to answer."
Keep answers concise and factual.
"""

human_prompt = """
Context:
{context}

User Question:
{question}
"""

prompt_template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(system_prompt),
    HumanMessagePromptTemplate.from_template(human_prompt),
])

# Build the conversational retrieval chain
rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    combine_docs_chain_kwargs={"prompt": prompt_template},
)

# Chat interface function
def chat_with_pdf(query: str):
    result = rag_chain.run({"question": query})
    return result



C:\Users\HP\AppData\Local\Temp\ipykernel_28188\2250896072.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


In [15]:
# Example interactive session

print("ChatPDF with Gemini is ready. Type 'exit' to quit.")
while True:
    user_input = input("You: ")
    if user_input.strip().lower() == "exit":
        break
    
    answer = chat_with_pdf(user_input)
    
    # Print user prompt and bot response clearly
    print(f"\nUser Prompt: {user_input}")
    print(f"Bot Response: {answer}\n")

ChatPDF with Gemini is ready. Type 'exit' to quit.

User Prompt: What the theme of the paper given
Bot Response: The paper is about identifying offensive content in social media posts and comments.


User Prompt: what is the theme of the paper
Bot Response: The paper lays a foundation for detecting offensive content in Nepali.


User Prompt: what are the offensive language classified in the paper
Bot Response: The offensive language classifications used in the paper are SEXIST, RACIST, and OTHER-OFFENSIVE.


User Prompt: where can i find which classification is done binary or multi
Bot Response: That information can be found in the section that discusses "coarse-grained (offensive or non-offensive) cases."


User Prompt: is there any other classification used 
Bot Response: Yes, Random Forrest classification was also used.


User Prompt: what category the offensive language is further divided into?
Bot Response: RACIST (OR), SEXIST(OS), and Other Offensive (OO) (e.g. attacktoanindividu